# Лабораторная работа 3. Стационарность и авторегрессия

**Цель.** Проверить стационарность ряда, применить преобразования и построить авторегрессионный прогноз.


## Ход работы

Ниже последовательно выполняются подготовка данных, построение графиков, расчёт признаков или моделей и краткая интерпретация результата. Случайные процедуры фиксируются через seed, чтобы результаты можно было повторить.


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)


## Настройка вычислений

Подключаются библиотеки, которые нужны для стационарности и авторегрессии. Фиксация случайности делает результаты повторяемыми при последующих запусках ноутбука.


In [ ]:
# Импортируем необходимые библиотеки

import os
from os import path
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.tsa.api as smt

## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
# загрузите данные о пассажирах
passengers = pd.read_csv('passengers.csv')
# неподходящий формат данных приводим к тому, с которым Pandas может работать
passengers['Month'] = pd.to_datetime(passengers['Month'])
# также устанавливаем индекс и сортируем
df = passengers.set_index('Month').sort_index()

## Первичная проверка

Здесь выводятся первые строки, размеры или базовая статистика. Такой просмотр нужен, чтобы убедиться, что данные загрузились без смещения колонок и что значения выглядят ожидаемо.


In [ ]:
df.describe()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
plt.plot(df["Passengers"])

## Проверка стационарности

Тест Дики-Фуллера используется как формальная проверка наличия единичного корня. Если ряд нестационарен, для моделей прогноза обычно нужны преобразования, разности или удаление тренда.


In [ ]:
# импортируем функцию, описывающую тест Дики-Фуллера
from statsmodels.tsa.stattools import adfuller

## Проверка стационарности

Тест Дики-Фуллера используется как формальная проверка наличия единичного корня. Если ряд нестационарен, для моделей прогноза обычно нужны преобразования, разности или удаление тренда.


In [ ]:
# всю теорию, описанную выше, реализуем с помощью statsmodels для проверки
# временного ряда перевозок на стационарность

alpha = 0.05
name = "Пассажиры"

# определяем временной ряд отдельной переменной
ts = df["Passengers"]

print(f'Тест Дики-Фуллера ряда {name} :')
# определяем результат значения теста из библиотеки с учетом
dftest = adfuller(ts, autolag='AIC')
dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])

for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
print(dfoutput)

if dfoutput["p-value"] < alpha:
    print(f"Значение p меньше {alpha * 100}%. Ряд стационарный.")
else:
    print(f"Значение p больше {alpha*100}%. Ряд не стационарный.")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# импортируем функцию seasonal_decompose из statsmodels
# (то есть осуществляем декомпозицию сигнала/временного ряда)
from statsmodels.tsa.seasonal import seasonal_decompose

# задаем размер графика
from pylab import rcParams
rcParams['figure.figsize'] = 11, 9


# примените функцию seasonal_decompose к данным о перевозках
decompose = seasonal_decompose(passengers["Passengers"],
                               period=6)
decompose.plot()
plt.show()

## Следующий этап расчёта

Этот блок продолжает основную цепочку стационарности и авторегрессии: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
nottrend = []
s = 6
notseason = []

# выборка без тренда
for i in range(1, len(df["Passengers"])):
   nottrend.append(df["Passengers"][i] - df["Passengers"][i-1])

# выборка без сезонности
for i in range(s, len(df["Passengers"])):
   notseason.append(df["Passengers"][i] - df["Passengers"][i-s])

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# отрисовываем временной ряд без тренда
plt.plot(nottrend)

## Проверка стационарности

Тест Дики-Фуллера используется как формальная проверка наличия единичного корня. Если ряд нестационарен, для моделей прогноза обычно нужны преобразования, разности или удаление тренда.


In [ ]:
alpha = 0.05
name = "Пассажиры без тренда"

ts = nottrend

print(f'Тест Дики-Фуллера ряда {name} :')
dftest = adfuller(ts, autolag='AIC')
dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])

for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
print(dfoutput)

if dfoutput["p-value"] < alpha:
    print(f"Значение p меньше {alpha * 100}%. Ряд стационарный.")
else:
    print(f"Значение p больше {alpha*100}%. Ряд не стационарный.")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# отрисовываем временной ряд без сезонности
plt.plot(notseason)

## Проверка стационарности

Тест Дики-Фуллера используется как формальная проверка наличия единичного корня. Если ряд нестационарен, для моделей прогноза обычно нужны преобразования, разности или удаление тренда.


In [ ]:
alpha = 0.05
name = "Пассажиры без сезона"

ts = notseason

print(f'Тест Дики-Фуллера ряда {name} :')
dftest = adfuller(ts, autolag='AIC')
dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])

for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
print(dfoutput)

if dfoutput["p-value"] < alpha:
    print(f"Значение p меньше {alpha * 100}%. Ряд стационарный.")
else:
    print(f"Значение p больше {alpha*100}%. Ряд не стационарный.")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# Преобразование Бокса-Кокса
from scipy.stats import boxcox

# вызываем функцию преобразования, которая выдает преобразованные данные и
# лучший параметр лямбда, который обеспечивает близость к нормальному
# распределению
transformed_data, best_lambda = boxcox(df["Passengers"])

# а теперь посмотрим на преобразованные данные
plt.plot(transformed_data)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
pnottrend = []

for i in range(1, len(transformed_data)):
   pnottrend.append(transformed_data[i] - transformed_data[i-1])


plt.plot(pnottrend)

## Проверка стационарности

Тест Дики-Фуллера используется как формальная проверка наличия единичного корня. Если ряд нестационарен, для моделей прогноза обычно нужны преобразования, разности или удаление тренда.


In [ ]:
alpha = 0.05
name = "Пассажиры после Кокса-Бокса"

ts = pnottrend

print(f'Тест Дики-Фуллера ряда {name} :')
dftest = adfuller(ts, autolag='AIC')
dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])

for key,value in dftest[4].items():
    dfoutput['Critical Value (%s)'%key] = value
print(dfoutput)

if dfoutput["p-value"] < alpha:
    print(f"Значение p меньше {alpha * 100}%. Ряд стационарный.")
else:
    print(f"Значение p больше {alpha*100}%. Ряд не стационарный.")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# AR(1)

N = 500

ar1 = [1]

for i in range(1, N):
    ar1.append(0.76 * ar1[i-1] + np.random.random())

plt.plot(ar1)

## Следующий этап расчёта

Этот блок продолжает основную цепочку стационарности и авторегрессии: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
print(f"standart deviation = {np.std(ar1)}\n mean = {np.mean(ar1)}")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
ts = pd.DataFrame(ar1)

fig = plt.figure(figsize=(20, 9))
layout = (2, 2)
ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
acf_ax = plt.subplot2grid(layout, (1, 0))
pacf_ax = plt.subplot2grid(layout, (1, 1))

ts.plot(ax=ts_ax)
ts_ax.set_title('Time Series Analysis Plots')
smt.graphics.plot_acf(ts, lags=20, ax=acf_ax, alpha=0.5)
smt.graphics.plot_pacf(ts, lags=10, ax=pacf_ax, alpha=0.5)
None

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# AR(1)

N = 500

ar2 = [1]

for i in range(1, N):
    ar2.append(- 0.76*ar2[i-1] + np.random.random())

plt.plot(ar2)

print(f"standart deviation = {np.std(ar2)}\n mean = {np.mean(ar2)}")

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
ts = pd.DataFrame(ar2)

fig = plt.figure(figsize=(20, 9))
layout = (2, 2)
ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
acf_ax = plt.subplot2grid(layout, (1, 0))
pacf_ax = plt.subplot2grid(layout, (1, 1))

ts.plot(ax=ts_ax)
ts_ax.set_title('Time Series Analysis Plots')
smt.graphics.plot_acf(ts, lags=20, ax=acf_ax, alpha=0.5)
smt.graphics.plot_pacf(ts, lags=10, ax=pacf_ax, alpha=0.5)
None

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# AR(1)

N = 500

ar3 = [1]

for i in range(1, N):
    ar3.append(2 * ar3[i-1] + np.random.random())

plt.plot(ar3)

print(f"standart deviation = {np.std(ar2)}\n mean = {np.mean(ar2)}")

## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
df = pd.read_csv('passengers.csv', names=["n","x"], skiprows=1)


df['t'] = df.index.values

ln = len(df)

# указываем 'объемы' выборок
train_cutoff = int(round(ln*0.75, 0))
validate_cutoff = int(round(ln*0.90,0))

# делим выборки
train_df = df[df['t'] <= train_cutoff]
validate_df = df[(df['t'] > train_cutoff) & (df['t'] <= validate_cutoff)]
forecast_df = df[df['t'] > validate_cutoff]

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
plt.plot(train_df.t, train_df.x, label='Training data')
plt.plot(validate_df.t, validate_df.x, label='Validate data')
plt.plot(forecast_df.t, forecast_df.x, label='For prediction')
plt.legend()
plt.title('Airline passengers by month')
plt.ylabel('Total passengers')
plt.xlabel('Month')
plt.show()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
from statsmodels.tsa.ar_model import AutoReg, ar_select_order

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
# создаем объект модели на основе данных временного ряда с 3 лагами
mod = AutoReg(df.t, 3, old_names=False)
# обучаем
res = mod.fit()

# выводим сводку информации об авторегрессионной модели
print(res.summary())

## Следующий этап расчёта

Этот блок продолжает основную цепочку стационарности и авторегрессии: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
# опять обучаем модель, но на этот раз указываем тип ковариационной оценки
res = mod.fit(cov_type="HC0")

# смотрим, что изменилось
print(res.summary())

## Следующий этап расчёта

Этот блок продолжает основную цепочку стационарности и авторегрессии: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
sel = ar_select_order(df.x, 13, old_names=False)
sel.ar_lags
res = sel.model.fit()
print(res.summary())

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
fig = res.plot_predict(train_cutoff)

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
pred = res.predict(start=0, end=train_cutoff, dynamic=False)
v_pred = res.predict(start=train_cutoff+1, end=(validate_cutoff), dynamic=False)
f_pred = res.predict(start=validate_cutoff + 1, end=(forecast_df.t[len(df.t)-1]), dynamic=False)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
plt.plot(train_df.t, train_df.x, label='Training data')
plt.plot(validate_df.t, validate_df.x, label='Validate data')
plt.plot(forecast_df.t, forecast_df.x, label='For prediction')
plt.plot(validate_df.t, v_pred, label='Validate prediction ')
plt.plot(forecast_df.t, f_pred, label='Forecast prediction')
plt.plot(train_df.t, pred, label='Train prediction')

plt.legend()
plt.title('Airline passengers by month')
plt.ylabel('Total passengers')
plt.xlabel('Month')
plt.show()

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
plt.plot(pred)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# MA

df['t'] = df.index.values

ln = len(df)

# указываем 'объемы' выборок
train_cutoff = int(round(ln*0.75, 0))
validate_cutoff = int(round(ln*0.90,0))

# делим выборки
train_df = df[df['t'] <= train_cutoff]
validate_df = df[(df['t'] > train_cutoff) & (df['t'] <= validate_cutoff)]
forecast_df = df[df['t'] > validate_cutoff]

plt.plot(train_df["t"], train_df["x"], label="original data")
plt.plot(train_df["t"], train_df["x"].rolling(10).mean(), label="MA")
plt.legend()
plt.ylabel('Total passengers')
plt.xlabel('Month')
plt.show()

## Оценка качества

Метрики переводят результат модели в численную форму. По ним можно сравнить разные подходы между собой и понять, где ошибка прогноза или классификации становится слишком большой.


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
print("RMSE:", np.sqrt(mean_squared_error(forecast_df.x, f_pred)))
print("MAPE:", mean_absolute_percentage_error(forecast_df.x, f_pred))
print("MAE:", mean_absolute_error(forecast_df.x, f_pred))
print("R2: ", r2_score(forecast_df.x, f_pred))

## Следующий этап расчёта

Этот блок продолжает основную цепочку стационарности и авторегрессии: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
# Простыми словами, нужно сделать задание по аналогии с тем нестационарным временным рядом (см. выше).

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
from statsmodels.tsa.arima_process import arma_generate_sample

np.random.seed(42)

# AR(2) процесс
ar_data = arma_generate_sample(
    ar=np.array([1.0, -0.5, 0.7]),
    ma=np.array([1]),
    nsample=200,
    scale=1,
    burnin=1000
)

time = np.arange(200)

trend = time * 0.2
seasonality = 2 * np.sin(2 * np.pi * time / 12)

time_series = trend + seasonality + ar_data

df = pd.DataFrame({
    "t": time,
    "x": time_series
})

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df.t, df.x, label="Original")
plt.title("Synthetic time series (trend + seasonality + AR)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Следующий этап расчёта

Этот блок продолжает основную цепочку стационарности и авторегрессии: результат предыдущих ячеек используется для следующего преобразования, визуализации или проверки. После выполнения стоит смотреть не только на числа, но и на то, как они связаны с исходным рядом.


In [ ]:
# ВАШ КОД ЗДЕСЬ! (почему всё ещё его не вижу???)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
df['t'] = df.index.values

ln = len(df)

# указываем 'объемы' выборок
train_cutoff = int(round(ln*0.75, 0))
validate_cutoff = int(round(ln*0.90,0))

# делим выборки
train_df = df[df['t'] <= train_cutoff]
validate_df = df[(df['t'] > train_cutoff) & (df['t'] <= validate_cutoff)]
forecast_df = df[df['t'] > validate_cutoff]

plt.plot(train_df.t, train_df.x, label='Training data')
plt.plot(validate_df.t, validate_df.x, label='Validate data')
plt.plot(forecast_df.t, forecast_df.x, label='For prediction')
plt.legend()
plt.title("Synthetic time series (trend + seasonality + AR)")
plt.ylabel('Total passengers')
plt.xlabel('Month')
plt.show()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
# создаем объект модели на основе данных временного ряда с 3 лагами
mod = AutoReg(df.t, 3, old_names=False)
# обучаем
res = mod.fit()

# выводим сводку информации об авторегрессионной модели
print(res.summary())

# опять обучаем модель, но на этот раз указываем тип ковариационной оценки
res = mod.fit(cov_type="HC0")

# смотрим, что изменилось
print(res.summary())

sel = ar_select_order(df.x, 13, old_names=False)
sel.ar_lags
res = sel.model.fit()
print(res.summary())

fig = res.plot_predict(train_cutoff)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
pred = res.predict(start=0, end=train_cutoff, dynamic=False)
v_pred = res.predict(start=train_cutoff+1, end=(validate_cutoff), dynamic=False)
f_pred = res.predict(start=validate_cutoff + 1, end=(forecast_df.t[len(df.t)-1]), dynamic=False)

plt.plot(train_df.t, train_df.x, label='Training data')
plt.plot(validate_df.t, validate_df.x, label='Validate data')
plt.plot(forecast_df.t, forecast_df.x, label='For prediction')
plt.plot(validate_df.t, v_pred, label='Validate prediction ')
plt.plot(forecast_df.t, f_pred, label='Forecast prediction')
plt.plot(train_df.t, pred, label='Train prediction')

plt.legend()
plt.title("Synthetic time series (trend + seasonality + AR)")
plt.ylabel('Total passengers')
plt.xlabel('Month')
plt.show()

plt.plot(pred)

## Визуальный анализ

График помогает увидеть форму ряда до формальных тестов: тренд, сезонность, шум, выбросы или различия между исходным и преобразованным сигналом. По нему удобно сверять, что дальнейшая обработка меняет ряд именно так, как ожидается.


In [ ]:
# MA

df['t'] = df.index.values

ln = len(df)

# указываем 'объемы' выборок
train_cutoff = int(round(ln*0.75, 0))
validate_cutoff = int(round(ln*0.90,0))

# делим выборки
train_df = df[df['t'] <= train_cutoff]
validate_df = df[(df['t'] > train_cutoff) & (df['t'] <= validate_cutoff)]
forecast_df = df[df['t'] > validate_cutoff]

plt.plot(train_df["t"], train_df["x"], label="original data")
plt.plot(train_df["t"], train_df["x"].rolling(5).mean(), label="MA")
plt.legend()
plt.ylabel('Total passengers')
plt.xlabel('Month')
plt.show()

## Построение прогноза

В этом блоке модель использует прошлые значения ряда для оценки будущих точек. После прогноза важно сравнить предсказание с тестовым участком и оценить, насколько модель улавливает динамику.


In [ ]:
print("RMSE:", np.sqrt(mean_squared_error(forecast_df.x, f_pred)))
print("MAPE:", mean_absolute_percentage_error(forecast_df.x, f_pred))
print("MAE:", mean_absolute_error(forecast_df.x, f_pred))
print("R2: ", r2_score(forecast_df.x, f_pred))
smape = 100 * np.mean(2 * np.abs(forecast_df.x - f_pred) / (np.abs(forecast_df.x) + np.abs(f_pred)))
print("SMAPE:", smape)

## Вывод

В работе показан полный путь от подготовки временного ряда до визуализации, расчёта признаков или построения модели. Основные результаты следует оценивать по графикам и метрикам, полученным при запуске ноутбука.
